# AgriLink — Recency-Weighted Recommendation

# AgriLink Customer Recommendation
This notebook is a standalone baseline. It runs with generated example data by default.
To use AgriLink data, set `USE_CSV = True` and update `CSV_PATH`. Expected columns are documented below.
Do not report results from synthetic data as real AgriLink performance.

**Goal:** prioritize products purchased recently, while still considering repeated purchases.
**Required real-data columns:** `CustomerID`, `OrderID`, `Product`, `Quantity`, `OrderDate`.

In [ ]:
import pandas as pd
import numpy as np

USE_CSV = False
CSV_PATH = "/content/agrilink_orders.csv"
rng = np.random.default_rng(7)
products = ["Apple","Banana","Beans","Brinjal","Cabbage","Carrot","Coriander",
            "Cucumber","Green Chilli","Onion","Orange","Papaya","Potato","Rice","Tomato"]
rows = []
for c in range(1, 61):
    for o in range(1, int(rng.integers(4, 9))):
        date = pd.Timestamp("2025-01-01") + pd.Timedelta(days=o*int(rng.integers(5, 14)))
        for p in rng.choice(products, size=int(rng.integers(2, 5)), replace=False):
            rows.append({"CustomerID":c, "OrderID":f"{c}_{o}", "Product":p,
                         "Quantity":int(rng.integers(1,4)), "OrderDate":date})
df = pd.DataFrame(rows)
if USE_CSV:
    df = pd.read_csv(CSV_PATH)
    missing = {"CustomerID","OrderID","Product","Quantity","OrderDate"} - set(df.columns)
    if missing: raise ValueError(f"CSV is missing columns: {missing}")
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
df.head()

## Score products by recency and frequency
For every past purchase line, compute age in days relative to the prediction date. A purchase contributes `Quantity × exp(-decay × age_days)`. Larger `decay` values make older purchases lose influence faster.

In [ ]:
def recency_recommendations(customer_id, df, top_n=5, decay=0.03, as_of=None,
                            exclude_previously_bought=False):
    history = df[df["CustomerID"] == customer_id].copy()
    if history.empty:
        return pd.DataFrame(columns=["Product","RecencyScore","LastPurchased"])
    if as_of is None:
        as_of = df["OrderDate"].max() + pd.Timedelta(days=1)
    as_of = pd.Timestamp(as_of)
    history["AgeDays"] = (as_of - history["OrderDate"]).dt.total_seconds() / 86400
    history["WeightedScore"] = history["Quantity"].clip(lower=0) * np.exp(
        -decay * history["AgeDays"].clip(lower=0)
    )
    scored = history.groupby("Product").agg(
        RecencyScore=("WeightedScore","sum"),
        LastPurchased=("OrderDate","max")
    ).reset_index()
    if exclude_previously_bought:
        # For a strict discovery feed, exclude all historically purchased products.
        scored = scored.iloc[0:0]
    return scored.sort_values(["RecencyScore","LastPurchased"], ascending=False).head(top_n)

customer_id = df["CustomerID"].iloc[0]
display(recency_recommendations(customer_id, df, top_n=10))

## Compare ordinary frequency with recency weighting
This comparison helps test whether recency changes the ordering of products.

In [ ]:
def frequency_scores(customer_id, df):
    x = df[df["CustomerID"] == customer_id]
    return (x.groupby("Product")
             .agg(PurchaseOrders=("OrderID","nunique"), TotalQuantity=("Quantity","sum"))
             .sort_values(["PurchaseOrders","TotalQuantity"], ascending=False))

print("Frequency baseline")
display(frequency_scores(customer_id, df).head(10))
print("Recency-weighted ranking")
display(recency_recommendations(customer_id, df, top_n=10, decay=0.03))

## Tune the decay parameter
Try multiple decay values and evaluate each on a chronological held-out basket. Do not choose a decay value based on the test set. This cell only demonstrates how the rankings change.

In [ ]:
for decay in [0.005, 0.01, 0.03, 0.07]:
    print(f"decay={decay}")
    display(recency_recommendations(customer_id, df, top_n=5, decay=decay))

## Limitations
- A real recency model requires trustworthy timestamps.
- A single global decay rate may not suit every product; perishables and staples have different replenishment cycles.
- The output is a ranking signal, not proof that a customer needs to reorder today.